# 🌍 01 — Data Download & Preprocessing
> **Hydrogeological Potential Mapping Toolkit**
> Author : HAMIDOU BÂ Abdoul Aziz | License : MIT

Ce notebook prépare l'environnement, télécharge automatiquement les limites administratives (GADM) et les séries pluviométriques CHIRPS, et vérifie l'intégrité des couches locales (BGS, MNT).


In [ ]:
from pathlib import Path
import yaml
import sys

# Ajouter le dossier src au path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT))

config_file = ROOT / 'config' / 'config_niger.yaml'
with open(config_file, 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

print(f"[OK] Configuration chargée pour : {cfg['study_area']['country_name']}")
print(f"     CRS : {cfg['study_area']['target_crs']} | Résolution : {cfg['study_area']['resolution']} m")


## 1. Initialisation des dossiers locaux


In [ ]:
DATA_RAW = ROOT / cfg['paths']['raw_dir']
DATA_PROC = ROOT / cfg['paths']['processed_dir']
MAPS = ROOT / cfg['paths']['output_maps_dir']

for folder in [
    DATA_RAW, DATA_PROC, MAPS,
    DATA_RAW / 'gadm', DATA_RAW / 'bgs',
    DATA_RAW / 'chirps', DATA_RAW / 'srtm', DATA_RAW / 'wpdx'
]:
    folder.mkdir(parents=True, exist_ok=True)
print('[OK] Arborescence vérifiée.')


## 2. Téléchargement automatique des limites GADM


In [ ]:
from src.hydromap.downloader import download_gadm_country

country_iso = cfg['study_area']['country_code']
gadm_path = download_gadm_country(country_iso, DATA_RAW / 'gadm')
print(f'Fichier GADM prêt : {gadm_path}')


## 3. Téléchargement des rasters de pluie CHIRPS


In [ ]:
from src.hydromap.downloader import download_chirps_annual

recent_years = [2020, 2021, 2022]
chirps_files = download_chirps_annual(recent_years, DATA_RAW / 'chirps')
print(f'{len(chirps_files)} fichiers CHIRPS téléchargés / vérifiés.')


## 4. Vérification des données hydrogéologiques (BGS) et d'altitude (MNT)


In [ ]:
bgs_path = ROOT / cfg['paths']['bgs_path']
dem_path = ROOT / cfg['paths']['dem_path']

print(f'Hydrogéologie BGS : {"Prêt" if bgs_path.exists() else "Manquant"} -> {bgs_path}')
print(f'MNT SRTM/HydroSHEDS: {"Prêt" if dem_path.exists() else "Manquant"} -> {dem_path}')

if not bgs_path.exists():
    print('\n👉 BGS Africa Groundwater Atlas : Téléchargez le shapefile sur https://www.bgs.ac.uk/africagroundwateratlas/')
if not dem_path.exists():
    print('👉 MNT : Téléchargez le MNT 30s ou 3s sur https://www.hydrosheds.org/hydrosheds-core-downloads')
